In [ ]:
from openai import OpenAI
import os
token = '' # Add your token here 
client = OpenAI(api_key=token)

import pymupdf
import os


def summarization_gen(instruction, model_temp, model_name):
    response = client.chat.completions.create(
        model=model_name,  # Or the specific model you have access to
        messages=[
            {"role": "system", "content": "You are a scientific assistant specializing in molecular biology and bioinformatics. Your goal is to generate concise, structured gene descriptions based on extracted scientific data."},
            {"role": "user", "content": f"{instruction}"}
        ],
    )
    
    # Extract the response
    # print(response)
    information = response.choices[0].message.content.strip()
    
    return information



def create_prompt_from_text(extracted_info_json):
    instruction = f"""
## **Task Description**
Generate a concise, well-structured gene description using the extracted gene facts below. Follow the style and structure of the example provided.

```
The extracted facts have already been assigned appropriate biological relationships and evidence status. When generating the summary, preserve these relationship phrases and evidence qualifiers. Summarize or combine related terms when appropriate, but do not change the meaning or evidence status of the extracted facts.

## **Example Gene Description**:
Predicted to enable heparin binding activity and transition metal ion binding activity. Involved in body morphogenesis; ecdysis, collagen and cuticulin-based cuticle; and nematode larval development. Located in early endosome; neuron projection; and neuronal cell body. Is expressed in several structures, including epithelial cell; excretory cell; hermaphrodite gonad; neurons; and somatic nervous system. Used to study Alzheimer's disease. Human ortholog(s) of this gene implicated in APP-related cerebral amyloid angiopathy; Alzheimer's disease (multiple); cognitive disorder; epilepsy; and traumatic brain injury. Orthologous to several human genes including APLP2 (amyloid beta precursor like protein 2) and APP (amyloid beta precursor protein).

## **Sentence Templates for Gene Description**:
- **Molecular Function**: Preserve the relationship phrase from the extracted facts, such as:
  - Enables <list of gene ontology molecular function terms>.
  - Predicted to enable <list of gene ontology molecular function terms>.
  - Contributes to <list of gene ontology molecular function terms>.
  - Predicted to contribute to <list of gene ontology molecular function terms>.

- **Biological Process**: Preserve the relationship phrase from the extracted facts, such as:
  - Involved in <list of biological processes>.
  - Predicted to be involved in <list of biological processes>.
  - Acts upstream of or within <list of biological processes>.
  - Predicted to act upstream of or within <list of biological processes>.

- **Subcellular Localization**: Preserve the relationship phrase from the extracted facts, such as:
  - Is active in <list of subcellular locations>.
  - Predicted to be active in <list of subcellular locations>.
  - Is located in <list of subcellular locations>.
  - Predicted to be located in <list of subcellular locations>.
  - Is part of <list of protein-containing complexes>.
  - Predicted to be part of <list of protein-containing complexes>.

- **Expression**: Preserve the relationship phrase from the extracted facts:
  - Is expressed in <list of anatomy ontology terms>.
  - Predicted to be expressed in <list of anatomy ontology terms>.

- **Pathways**: Preserve the relationship phrase from the extracted facts:
  - Is part of <list of pathway terms>.
  - Predicted to be part of <list of pathway terms>.

- **Interaction**: Preserve the relationship phrase from the extracted facts:
  - Interacts with <list of genes/proteins>.
  - Predicted to interact with <list of genes/proteins>.

- **Disease Relevance**:
  - Used to study <list of disease ontology terms>.

- **Implication of Human Orthologs in Disease**:
  - Human ortholog(s) of this gene implicated in <list of disease ontology terms>.

- **Human Orthologs**:
  - Orthologous to <list of human orthologs>.

- **Modifiers**:
  - <Modifier> ameliorates or exacerbates the disease phenotype or condition.

## **Rules for Summarization**
- **General Guidelines**:
  - Maintain a **scientific tone** and avoid redundant phrases.
  - Ensure **grammatical correctness**.
  - Use **semicolons (;)** for separating multiple terms when appropriate.
  - **Omit** fields with no available data.
  - Summarize multiple related terms into higher-level categories when appropriate to avoid excessive listing.

- **Relationship and Evidence Preservation**:
  - **Preserve the relationship phrase and evidence status of every extracted fact.**
  - If an extracted fact uses **"predicted"**, the corresponding statement in the final summary must also use **"predicted"**.
  - Never convert a predicted statement into an unqualified statement.
  - Never convert an unqualified statement into a predicted statement.
  - Preserve distinctions between different biological relationships. For example:
    - `enables` must not be changed to `contributes to`, and vice versa.
    - `is involved in` must not be changed to `acts upstream of or within`, and vice versa.
    - `is active in`, `is located in`, and `is part of` must remain distinct.
  - When multiple facts have the **same relationship phrase and evidence status**, they may be combined into one sentence.
  - When facts have **different relationship phrases or different evidence status**, keep them in separate statements.
  - Summarization may group or shorten the biological terms, but must not change their associated relationship or evidence status.

## **Extracted Information**:
{extracted_info_json}

Using this structured information, generate a complete and well-formatted gene description. Preserve the relationship phrase and evidence status from the extracted facts while combining compatible facts into concise sentences.
"""

    return instruction

In [ ]:
import os
import json
from openai import OpenAI
from pprint import pprint

with open('data/data_pairs_20251002.json', 'r') as f:
    gene_data = json.load(f)

models = ['o1'] #['gpt-4-turbo', 'o1', 'gpt-5', 'gpt-4o']
folder = 'gene_description/no_desc_2026'

for gene_name in gene_data.keys():
    if not gene_data[gene_name]['gt']['description'] or len(gene_data[gene_name]['gt']['description'])<5:
        print(gene_name)
        paper_info = {}
        paper_info['gene_name'] = gene_name
        for key in gene_data[gene_name]['paper_inf'].keys():
            if len(gene_data[gene_name]['paper_inf'][key])>0:
                print(gene_data[gene_name]['paper_inf'][key])
                for item in gene_data[gene_name]['paper_inf'][key]:
                    paper_info[key] = [item['sentence template'] for item in gene_data[gene_name]['paper_inf'][key] if 'sentence template' in item.keys()]
        instruction = create_prompt_from_text(paper_info)
        print(instruction) 
        model_temp = 0.0
        
        gene_info = {}
        for model_name in models:
            gene_info[model_name] = summarization_gen(instruction, model_temp, model_name)
        print('*'*10)
        print(gene_info)
        print('*'*10)
        task_name = gene_name+ '_NoDesc_paperinfo_20260927.json'
        with open(os.path.join(folder, task_name), 'w') as f:
            json.dump(gene_info, f, indent=4)